In [0]:
import os
import sys
from pyspark.sql.functions import current_timestamp, input_file_name, col

# Add the parent directory to Python path to enable imports
parent_path = '/Workspace/Users/esymphony.life@gmail.com/MSF_test/customer'
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

from common.functions import ingest_to_delta

In [0]:
# Define catalog, domain, and schema for ingestion
catalog = "customer"
domain = "transaction"
schema = "bronze"

In [0]:

def get_uningested_files(volume_path, schema="bronze"):
    """
    Identifies all files in a Volume that have not yet been ingested into 
    the corresponding table (derived from the volume path).
    """
    # Parse volume_path to derive the catalog.schema.table
    _, _, catalog, domain, table_name = volume_path.split("/")
    target_table = f"{catalog}.{domain}.{schema}_{table_name}"   

    # Get current files in the volume
    current_files_df = (spark.read.format("binaryFile")
                        .option("recursiveFileLookup", "true")
                        .load(volume_path)
                        .select(col("path"), col("modificationTime")))

    # Check against the derived target table
    if spark.catalog.tableExists(target_table):
        # Select distinct source files already processed
        ingested_df = (spark.read.table(target_table)
                       .select(col("_source_file").alias("path"))
                       .distinct())
        
        # Filter out already processed files
        unprocessed_df = current_files_df.join(ingested_df, on="path", how="left_anti")
    else:
        # Initial ingestion
        unprocessed_df = current_files_df

    unprocessed_rows = (unprocessed_df
                        .orderBy(col("modificationTime").asc())
                        .select("path")
                        .collect())

    # Return a list of strings
    return [row.path[5:] for row in unprocessed_rows]

def ingest_volume(volume, schema="bronze"):
    volume_path = f"/Volumes/{catalog}/{domain}/{volume}"

    write_path = f"{catalog}.{domain}.{schema}_{volume}"

    for file_path in get_uningested_files(volume_path):
        filetype = file_path.split(".")[-1]
        ingest_to_delta(spark, filetype, file_path, write_path)
    
    print(f"Successfully ingested all files from {volume}")


In [0]:
trx_vol = f"transaction_data"
ingest_volume(trx_vol)
